# Cross-Dataset Classification Matrix

Three output-free, reproducible evaluation blocks share capped parent cohorts.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from classification_utils import split_cohort
from cross_dataset_classification_utils import (
    DATASETS, EVALUATION_REGIMES, SCORER_LLMS, TRAINING_REGIMES,
    build_deterministic_parent_cohorts, build_shared_parent_manifest, concat_or_empty, evaluate_whole_text_datasets,
    combine_whole_text_dataset_results, label_result_frame, pivot_matrix, select_scalar_partition, source_coverage_row, summarize_bbc_false_positives, train_whole_text_models,
)
from local_evidence_evaluation_utils import (
    combine_local_evidence_dataset_results, evaluate_local_evidence_datasets, evaluate_local_evidence_validation_lengths,
    evaluate_local_evidence_semeval,
)
from local_evidence_training_utils import train_local_evidence_models
from local_evidence_utils import load_local_evidence_parent_metadata
from logprob_metrics_distributions_utils import load_logprob_rows
from whole_text_classification_utils import evaluate_whole_text_validation_lengths, load_whole_text_cohort

sns.set_theme(style="whitegrid", context="notebook")
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
DATASETS = {"Kaggle": "kaggle_train", "MAGE": "mage_train", "Outfox": "outfox_chatgpt_train", "BBC": "bbc_news_content"}
SOURCE_SPLITS = {"BBC": ("HG",)}
LLM_MODELS = ["gemma_3_1b_it", "gemma_3_270m_it", "littlelamb", "smollm2_360m"]
TRAINING_REGIMES = {"Kaggle": ("Kaggle",), "MAGE": ("MAGE",), "Outfox": ("Outfox",), "Mixed": ("Kaggle", "MAGE", "Outfox"), "Mixed+BBC": ("Kaggle", "MAGE", "Outfox", "BBC")}
EVALUATION_REGIMES = {"Kaggle": ("Kaggle",), "MAGE": ("MAGE",), "Outfox": ("Outfox",), "BBC": ("BBC",), "Mixed": ("Kaggle", "MAGE", "Outfox")}
CLASSIFIER_NAMES = ("logistic_regression",)
WINDOW_LENGTHS = (32, 64, 128)
STRIDE, TOP_K, SEED, MAX_PER_SPLIT = 16, 20, 42, 500
SEMEVAL_DATASETS = ("semeval2024_task8_subtask_c_train", "semeval2024_task8_subtask_c_dev")
SEMEVAL_PREPARED_ROOT = DATA_ROOT / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC"
VALIDATION_WORD_LIMITS = (50, 100, 200, 400)

In [ ]:
source_data, source_coverage = {}, []
for llm in LLM_MODELS:
    source_data[llm] = {}
    for source_name, dataset_alias in DATASETS.items():
        source_splits = SOURCE_SPLITS.get(source_name, ("HG", "MG"))
        try:
            scalar = load_whole_text_cohort(PROCESSED_ROOT, dataset=dataset_alias, scorer_llm=llm)
            local_parents = load_local_evidence_parent_metadata(PROCESSED_ROOT, dataset_alias, llm, window_lengths=WINDOW_LENGTHS, stride=STRIDE, top_k=TOP_K, splits=source_splits)
            manifest = build_shared_parent_manifest(scalar, local_parents, seed=SEED, max_per_split=MAX_PER_SPLIT)
            local_partitions = {partition: manifest[partition][["split", "text_id", "label"]] for partition in manifest}
            scalar_partitions = {partition: select_scalar_partition(scalar, manifest[partition]) for partition in manifest}
            source_data[llm][source_name] = {"alias": dataset_alias, "local": local_partitions, "scalar_partitions": scalar_partitions, "manifest": manifest}
            source_coverage.append({"llm": llm, "source_dataset": source_name, "status": "available", "classifier": "source_coverage", "rows": len(scalar)})
            del scalar, local_parents
        except (FileNotFoundError, KeyError, ValueError, OSError) as error:
            source_coverage.append(source_coverage_row(llm, source_name, error))
            continue

whole_text_metric_frames, whole_text_prediction_frames, whole_text_coverage_frames = [], [], [pd.DataFrame(source_coverage)]
whole_model_bundles, short_validation_frames = {}, []
local_evidence_metric_frames, local_evidence_prediction_frames, local_evidence_coverage_frames = [], [], [pd.DataFrame(source_coverage)]
semeval_region_frames, semeval_region_prediction_frames, semeval_boundary_frames, semeval_prediction_frames, semeval_exclusion_frames = [], [], [], [], []
semeval_evaluation_frames = []
local_model_bundles = {}

In [ ]:
# BLOCK 1: WHOLE-TEXT SCALAR METRICS
for llm, datasets in source_data.items():
    for training_regime, training_names in TRAINING_REGIMES.items():
        if not set(training_names).issubset(datasets):
            continue
        train_cohorts = {datasets[name]["alias"]: datasets[name]["scalar_partitions"]["train"] for name in training_names}
        validation_cohorts = {datasets[name]["alias"]: datasets[name]["scalar_partitions"]["validation"] for name in training_names}
        trained = train_whole_text_models(train_cohorts, validation_cohorts, scorer_llm=llm)
        whole_model_bundles[(llm, training_regime)] = trained.bundle
        if training_regime == "Mixed":
            stage1_rows = {datasets[name]["alias"]: load_logprob_rows(PROCESSED_ROOT, selected_datasets=[datasets[name]["alias"]], selected_models=[llm], selected_parent_ids={split: set(ids.astype("int64")) for split, ids in datasets[name]["manifest"]["validation"].groupby("split")["text_id"]}) for name in training_names}
            short_validation = evaluate_whole_text_validation_lengths(trained.bundle, validation_cohorts, stage1_rows, VALIDATION_WORD_LIMITS)
            short_validation_frames.append(label_result_frame(short_validation.metrics, llm=llm, training_regime=training_regime, evaluation_regime="Mixed", evaluation_dataset="Mixed", block="whole_text").assign(classifier_family="whole_text_statistical"))
            del stage1_rows
        for evaluation_regime, evaluation_names in EVALUATION_REGIMES.items():
            if not set(evaluation_names).issubset(datasets):
                continue
            for partition in ("train", "validation", "test"):
                separate_results = [evaluate_whole_text_datasets(trained.bundle, {datasets[name]["alias"]: datasets[name]["scalar_partitions"][partition]}, partition=partition) for name in evaluation_names]
                evaluated = combine_whole_text_dataset_results(separate_results, evaluation_regime=evaluation_regime) if evaluation_regime == "Mixed" else separate_results[0]
                for name, frame in (("metrics", evaluated.metrics), ("predictions", evaluated.predictions), ("coverage", evaluated.coverage)):
                    labelled = label_result_frame(frame, llm=llm, training_regime=training_regime, evaluation_regime=evaluation_regime, evaluation_dataset=("Mixed" if evaluation_regime == "Mixed" else ", ".join(evaluation_names)), block="whole_text")
                    if name == "metrics": whole_text_metric_frames.append(labelled)
                    elif name == "predictions": whole_text_prediction_frames.append(labelled)
                    else: whole_text_coverage_frames.append(labelled)

whole_text_metrics = concat_or_empty(whole_text_metric_frames)
whole_text_predictions = concat_or_empty(whole_text_prediction_frames)
whole_text_coverage = concat_or_empty(whole_text_coverage_frames)

In [ ]:
# BLOCK 2: LOCAL EVIDENCE
for llm, datasets in source_data.items():
    for training_regime, training_names in TRAINING_REGIMES.items():
        if not set(training_names).issubset(datasets): continue
        aliases = tuple(datasets[name]["alias"] for name in training_names)
        trained = train_local_evidence_models(PROCESSED_ROOT, aliases, llm, {datasets[name]["alias"]: datasets[name]["local"]["train"] for name in training_names}, {datasets[name]["alias"]: datasets[name]["local"]["validation"] for name in training_names}, classifier_names=CLASSIFIER_NAMES, window_lengths=WINDOW_LENGTHS, stride=STRIDE, top_k=TOP_K, seed=SEED)
        local_model_bundles[(llm, training_regime)] = trained.model
        if training_regime == "Mixed":
            stage1_rows = {datasets[name]["alias"]: load_logprob_rows(PROCESSED_ROOT, selected_datasets=[datasets[name]["alias"]], selected_models=[llm], selected_parent_ids={split: set(ids.astype("int64")) for split, ids in datasets[name]["manifest"]["validation"].groupby("split")["text_id"]}) for name in training_names}
            short_validation = evaluate_local_evidence_validation_lengths(trained.model, PROCESSED_ROOT, aliases, {datasets[name]["alias"]: datasets[name]["local"]["validation"] for name in training_names}, stage1_rows, VALIDATION_WORD_LIMITS)
            short_validation_frames.append(label_result_frame(short_validation.metrics, llm=llm, training_regime=training_regime, evaluation_regime="Mixed", evaluation_dataset="Mixed", block="local_evidence").assign(classifier_family="local_evidence_agg_mg_score"))
            del stage1_rows
        for evaluation_regime, evaluation_names in EVALUATION_REGIMES.items():
            if not set(evaluation_names).issubset(datasets): continue
            for partition in ("train", "validation", "test"):
                separate_results = [evaluate_local_evidence_datasets(trained.model, PROCESSED_ROOT, (datasets[name]["alias"],), {datasets[name]["alias"]: datasets[name]["local"][partition]}) for name in evaluation_names]
                evaluated = combine_local_evidence_dataset_results(separate_results, evaluation_regime=evaluation_regime) if evaluation_regime == "Mixed" else separate_results[0]
                for name, frame in (("metrics", evaluated.metrics), ("predictions", evaluated.predictions), ("coverage", evaluated.coverage)):
                    labelled = label_result_frame(frame, llm=llm, training_regime=training_regime, evaluation_regime=evaluation_regime, evaluation_dataset=("Mixed" if evaluation_regime == "Mixed" else ", ".join(evaluation_names)), block="local_evidence").assign(partition=partition)
                    if name == "metrics": local_evidence_metric_frames.append(labelled)
                    elif name == "predictions": local_evidence_prediction_frames.append(labelled)
                    else: local_evidence_coverage_frames.append(labelled)

local_evidence_metrics = concat_or_empty(local_evidence_metric_frames)
local_evidence_predictions = concat_or_empty(local_evidence_prediction_frames)
local_evidence_coverage = concat_or_empty(local_evidence_coverage_frames)

In [ ]:
# BLOCK 3: SEMEVAL BOUNDARIES
for llm, datasets in source_data.items():
    for training_regime, training_names in TRAINING_REGIMES.items():
        if not set(training_names).issubset(datasets): continue
        aliases = tuple(datasets[name]["alias"] for name in training_names)
        trained = local_model_bundles[(llm, training_regime)]
        for semeval_dataset in SEMEVAL_DATASETS:
            semeval_path = SEMEVAL_PREPARED_ROOT / ("SemEval2024_task8_C_train_MG.csv" if semeval_dataset.endswith("_train") else "SemEval2024_task8_C_dev_MG.csv")
            try:
                semeval_annotations = pd.read_csv(semeval_path)
                semeval = evaluate_local_evidence_semeval(trained, PROCESSED_ROOT, semeval_annotations, dataset=semeval_dataset, split="MG", sustained_length=3)
                semeval_region_frames.append(label_result_frame(semeval.summary, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_region"))
                semeval_region_prediction_frames.append(label_result_frame(semeval.predicted_regions, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_region_detail"))
                semeval_evaluation_frames.append(label_result_frame(semeval.evaluations, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_region_detail"))
                semeval_boundary_frames.append(label_result_frame(semeval.boundary_summary, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_boundary"))
                semeval_prediction_frames.append(label_result_frame(semeval.boundary_predictions, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_boundary"))
                semeval_exclusion_frames.append(label_result_frame(semeval.boundary_exclusions, llm=llm, training_regime=training_regime, evaluation_regime="SemEval", evaluation_dataset=semeval_dataset, block="semeval_boundary"))
            except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                semeval_exclusion_frames.append(pd.DataFrame([{"llm": llm, "training_regime": training_regime, "evaluation_dataset": semeval_dataset, "reason": str(error)}]))

semeval_region_summary = concat_or_empty(semeval_region_frames)
semeval_predicted_regions = concat_or_empty(semeval_region_prediction_frames)
semeval_region_evaluations = concat_or_empty(semeval_evaluation_frames)
semeval_boundary_summary = concat_or_empty(semeval_boundary_frames)
boundary_predictions = concat_or_empty(semeval_prediction_frames)
boundary_exclusions = concat_or_empty(semeval_exclusion_frames)

In [ ]:
display(whole_text_coverage, whole_text_metrics, whole_text_predictions, local_evidence_coverage, local_evidence_metrics, local_evidence_predictions, semeval_region_summary, semeval_predicted_regions, semeval_region_evaluations, semeval_boundary_summary, boundary_predictions, boundary_exclusions)
# BBC one-class evaluation is valid for Task 1 metrics; binary ROC-AUC/PR-AUC are not meaningful there.
for block, frame, value, index, columns in (("whole_text", whole_text_metrics, "macro_f1", ["llm", "training_regime", "classifier", "partition", "feature_set"], ["evaluation_regime", "evaluation_dataset"]), ("local_evidence", local_evidence_metrics, "macro_f1", ["llm", "training_regime", "classifier", "partition", "aggregation", "window_length"], ["evaluation_regime", "evaluation_dataset"]), ("semeval_region", semeval_region_summary, "mg_f1", ["llm", "training_regime", "classifier", "window_length"], ["evaluation_dataset"]), ("semeval_boundary", semeval_boundary_summary, "mean_absolute_token_error", ["llm", "training_regime", "classifier", "window_length"], ["evaluation_dataset"])):
    if frame.empty: continue
    matrix = pivot_matrix(frame, value=value, index=index, columns=columns)
    display(matrix)
    sns.heatmap(matrix, annot=True, fmt=".3f", mask=matrix.isna(), vmin=0, vmax=1, cmap="viridis")
    plt.title(f"{block} matrix")
    plt.show()

In [ ]:
# ADDITIONAL VISUALIZATIONS: TRAIN/VALIDATION/TEST MATRICES AND SEMEVAL BOUNDARIES
for block, frame in (("Scalar metrics", whole_text_metrics), ("Local evidence whole text", local_evidence_metrics)):
    if frame.empty:
        continue
    configuration_columns = [column for column in ("classifier", "model", "aggregation", "window_length", "feature_set") if column in frame]
    for llm in sorted(frame["llm"].dropna().unique()):
        llm_frame = frame.loc[frame["llm"].eq(llm)]
        configurations = llm_frame.groupby(configuration_columns, dropna=False, sort=True) if configuration_columns else [((), llm_frame)]
        for configuration, configuration_frame in configurations:
            configuration_label = " | ".join(f"{column}={value}" for column, value in zip(configuration_columns, configuration if isinstance(configuration, tuple) else (configuration,)))
            for partition in ("train", "validation", "test"):
                selected = configuration_frame.loc[configuration_frame["partition"].eq(partition)]
                if selected.empty:
                    continue
                matrix = selected.pivot_table(index="training_regime", columns="evaluation_regime", values="macro_f1", aggfunc="mean")
                if matrix.empty:
                    continue
                plt.figure(figsize=(max(6, 1.5 * len(matrix.columns)), max(4, 1.0 * len(matrix.index))))
                sns.heatmap(matrix, annot=True, fmt=".3f", vmin=0, vmax=1, cmap="viridis")
                plt.title(f"{block} macro F1 | {llm} | {partition} | {configuration_label}")
                plt.xlabel("Evaluation dataset/regime")
                plt.ylabel("Training dataset/regime")
                plt.tight_layout()
                plt.show()

if not semeval_boundary_summary.empty:
    boundary_metrics = ["exact_accuracy", "within_1_accuracy", "within_3_accuracy", "within_5_accuracy"]
    boundary_plot = semeval_boundary_summary.melt(id_vars=["llm", "training_regime", "evaluation_dataset", "window_length"], value_vars=boundary_metrics, var_name="accuracy_tolerance", value_name="accuracy")
    boundary_grid = sns.catplot(data=boundary_plot, x="training_regime", y="accuracy", hue="accuracy_tolerance", col="evaluation_dataset", row="llm", kind="bar", sharey=True, height=4, aspect=1.4)
    boundary_grid.set_axis_labels("Training dataset", "Boundary accuracy")
    boundary_grid.set_titles("{row_name} | {col_name}")
    plt.show()

if not boundary_predictions.empty:
    sample_columns = {"llm", "training_regime", "evaluation_dataset", "window_length", "text_id", "token_count", "predicted_token_boundary", "gold_token_boundary"}
    if sample_columns.issubset(boundary_predictions.columns):
        samples = boundary_predictions.sort_values(["llm", "training_regime", "evaluation_dataset", "text_id", "window_length"]).drop_duplicates(["llm", "training_regime", "evaluation_dataset", "text_id"])
        samples = samples.groupby(["llm", "training_regime", "evaluation_dataset"], sort=True, group_keys=False).head(3)
        for (llm, training_regime, dataset), group in samples.groupby(["llm", "training_regime", "evaluation_dataset"], sort=True):
            figure, axes = plt.subplots(len(group), 1, figsize=(12, max(2.5, 2.5 * len(group))), squeeze=False)
            for axis, (_, row) in zip(axes[:, 0], group.iterrows()):
                axis.axvspan(row["gold_token_boundary"], row["token_count"], color="tab:green", alpha=0.12, label="gold MG span")
                axis.axvspan(row["predicted_token_boundary"], row["token_count"], color="tab:red", alpha=0.12, label="predicted MG span")
                axis.axvline(row["gold_token_boundary"], color="tab:green", label="gold boundary")
                axis.axvline(row["predicted_token_boundary"], color="tab:red", linestyle="--", label="predicted boundary")
                axis.set_xlim(0, row["token_count"])
                axis.set_yticks([])
                axis.set_title(f"{llm} | {training_regime} | {dataset} | text {row['text_id']} | window {row['window_length']}")
                axis.legend(loc="upper right", fontsize="small")
            figure.supxlabel("Token position")
            figure.tight_layout()
            plt.show()

short_validation_comparison = concat_or_empty(short_validation_frames)
display(short_validation_comparison)
if not short_validation_comparison.empty:
    short_validation_comparison = short_validation_comparison.assign(word_limit_label=short_validation_comparison["word_limit"].map(lambda limit: "none_limit" if pd.isna(limit) else f"{int(limit)}_limit"))
    short_plot = sns.catplot(data=short_validation_comparison, x="word_limit_label", y="macro_f1", hue="classifier_family", col="llm", kind="bar", height=4, aspect=1.3)
    short_plot.set_axis_labels("Validation prefix word limit", "Validation macro F1")
    short_plot.set_titles("{col_name} | Mixed training")
    plt.show()

# MIXED-SOURCE LLM COMPARISON
for block, frame in (("Scalar metrics", whole_text_metrics), ("Local evidence", local_evidence_metrics)):
    mixed = frame.loc[frame["training_regime"].eq("Mixed")] if not frame.empty else frame
    if mixed.empty:
        continue
    comparison = sns.catplot(data=mixed, x="llm", y="macro_f1", hue="evaluation_regime", col="partition", kind="bar", height=4, aspect=1.3)
    comparison.set_axis_labels("Scorer LLM", "Macro F1")
    comparison.set_titles(f"{block} | Mixed training | {{col_name}}")
    comparison.set_xticklabels(rotation=30)
    plt.show()

bbc_false_positive_summary = concat_or_empty([
    summarize_bbc_false_positives(whole_text_predictions, block="whole_text"),
    summarize_bbc_false_positives(local_evidence_predictions, block="local_evidence"),
])
display(bbc_false_positive_summary)
if not bbc_false_positive_summary.empty:
    bbc_plot = sns.catplot(data=bbc_false_positive_summary, x="llm", y="false_hg_as_mg", hue="block", col="partition", kind="bar", height=4, aspect=1.3)
    bbc_plot.set_axis_labels("Scorer LLM", "HG texts predicted as MG")
    bbc_plot.set_xticklabels(rotation=30)
    plt.show()

mixed_boundary = semeval_boundary_summary.loc[semeval_boundary_summary["training_regime"].eq("Mixed")] if not semeval_boundary_summary.empty else semeval_boundary_summary
if not mixed_boundary.empty:
    mixed_boundary_plot = mixed_boundary.melt(id_vars=["llm", "evaluation_dataset", "window_length"], value_vars=["exact_accuracy", "within_1_accuracy", "within_3_accuracy", "within_5_accuracy"], var_name="accuracy_tolerance", value_name="accuracy")
    comparison = sns.catplot(data=mixed_boundary_plot, x="llm", y="accuracy", hue="accuracy_tolerance", col="evaluation_dataset", row="window_length", kind="bar", height=4, aspect=1.4)
    comparison.set_axis_labels("Scorer LLM", "Boundary accuracy")
    comparison.set_xticklabels(rotation=30)
    plt.show()